# 01 · Regresión y clasificación: de modelos lineales a decisiones probabilísticas

Los modelos lineales siguen siendo esenciales: son rápidos, interpretables y buenos baselines. Este lab cubre regresión lineal, Ridge/Lasso/ElasticNet, regresión logística, diagnóstico de residuos, regularización y evaluación.

## Objetivos
- Entender la ecuación lineal $\hat y=X\beta+b$.
- Relacionar mínimos cuadrados con MSE.
- Ver cómo L1 y L2 controlan complejidad.
- Pasar de una salida lineal a probabilidad mediante la función sigmoide.
- Evaluar regresión y clasificación con métricas apropiadas.
- Interpretar coeficientes y detectar supuestos problemáticos.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
SEED=42

## 1. Regresión lineal

La regresión estima una variable continua. Los coeficientes representan el cambio esperado de la predicción cuando una feature aumenta una unidad, manteniendo las demás constantes. Esa interpretación requiere cuidado cuando hay escalas distintas, colinealidad o transformaciones.


In [ ]:
X,y=load_diabetes(return_X_y=True,as_frame=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=SEED)
models={
 'Linear':LinearRegression(),
 'Ridge':Ridge(alpha=1.0),
 'Lasso':Lasso(alpha=.05,max_iter=20000),
 'ElasticNet':ElasticNet(alpha=.01,l1_ratio=.5,max_iter=20000)
}
rows=[]
for name,m in models.items():
    m.fit(Xtr,ytr); p=m.predict(Xte)
    rows.append([name,mean_absolute_error(yte,p),mean_squared_error(yte,p)**.5,r2_score(yte,p)])
pd.DataFrame(rows,columns=['modelo','MAE','RMSE','R2']).sort_values('RMSE').round(3)

### L1 vs L2
- **Ridge/L2:** $\lambda\sum_j\beta_j^2$. Reduce coeficientes suavemente; útil con colinealidad.
- **Lasso/L1:** $\lambda\sum_j|\beta_j|$. Puede llevar coeficientes exactamente a cero y actuar como selección de variables.
- **ElasticNet:** combina ambas.

Siempre conviene estandarizar cuando la penalización depende de magnitudes de coeficientes.


In [ ]:
alphas=np.logspace(-4,2,30)
coef=[]
for a in alphas:
    pipe=make_pipeline(StandardScaler(),Ridge(alpha=a))
    pipe.fit(Xtr,ytr); coef.append(pipe[-1].coef_)
plt.semilogx(alphas,np.array(coef))
plt.xlabel('alpha'); plt.ylabel('coeficiente'); plt.title('Ruta de regularización Ridge'); plt.show()

## 2. Diagnóstico de residuos

Un buen ajuste no debería dejar patrones sistemáticos en los residuos. Curvaturas pueden indicar no linealidad; forma de embudo sugiere heteroscedasticidad; outliers extremos pueden dominar MSE.


In [ ]:
m=Ridge(alpha=1).fit(Xtr,ytr); pred=m.predict(Xte); resid=yte-pred
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].scatter(pred,resid,alpha=.7); ax[0].axhline(0,color='black'); ax[0].set(xlabel='predicción',ylabel='residuo',title='Residuos vs predicción')
ax[1].hist(resid,bins=20); ax[1].set_title('Distribución de residuos'); plt.show()

## 3. Relaciones no lineales sin abandonar modelos lineales

`PolynomialFeatures` crea $x^2$, interacciones $x_1x_2$, etc. Aumenta expresividad, pero también riesgo de overfitting: por eso suele combinarse con regularización.


In [ ]:
rng=np.random.default_rng(SEED); x=np.linspace(-3,3,120)[:,None]; y2=2*x[:,0]**2-x[:,0]+rng.normal(0,1,120)
for degree in [1,2,8]:
    pipe=make_pipeline(PolynomialFeatures(degree),StandardScaler(),Ridge(alpha=1))
    pipe.fit(x,y2); plt.plot(x[:,0],pipe.predict(x),label=f'grado {degree}')
plt.scatter(x[:,0],y2,s=12,alpha=.45); plt.legend(); plt.title('Bias/variance con features polinomiales'); plt.show()

## 4. Regresión logística = clasificación probabilística

La logística modela $P(y=1|x)=\sigma(z)$, donde $\sigma(z)=1/(1+e^{-z})$. El umbral 0.5 **no es una ley**: se elige según costos de falso positivo/falso negativo.


In [ ]:
X,y=load_breast_cancer(return_X_y=True,as_frame=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEED,stratify=y)
clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=5000,C=1.0))
clf.fit(Xtr,ytr); proba=clf.predict_proba(Xte)[:,1]
for th in [.3,.5,.7]:
    pred=(proba>=th).astype(int)
    print('\nUMBRAL',th); print(classification_report(yte,pred,digits=3))

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(15,4))
ConfusionMatrixDisplay.from_predictions(yte,(proba>=.5).astype(int),ax=ax[0],colorbar=False)
RocCurveDisplay.from_predictions(yte,proba,ax=ax[1])
PrecisionRecallDisplay.from_predictions(yte,proba,ax=ax[2])
plt.tight_layout(); plt.show()

## ¿Cuándo usar estos modelos?
**Regresión lineal/Ridge:** baseline, interpretación, relaciones aproximadamente lineales, datasets tabulares pequeños/medianos.
**Lasso/ElasticNet:** muchas variables y necesidad de regularización/selección.
**Logística:** scoring de riesgo, clasificación binaria interpretable, probabilidades calibrables.

## Errores comunes
- evaluar solo R² o accuracy;
- interpretar correlación como causalidad;
- no escalar antes de regularizar;
- seleccionar el threshold usando test;
- ignorar outliers y multicolinealidad;
- crear features polinomiales de alto grado sin regularización.

## Ejercicios
1. Calcula VIF o matriz de correlación y estudia multicolinealidad.
2. Compara Ridge/Lasso con validación cruzada para distintos `alpha`.
3. Calcula PR-AUC y ROC-AUC para logística.
4. Define una matriz de costos: falso negativo cuesta 5 veces más. Encuentra el mejor threshold.
5. Implementa regresión lineal con NumPy usando descenso de gradiente.
6. Crea un caso sintético donde una transformación logarítmica mejore el modelo.
